In [2]:
#Join and features
# Reads: ../data/clean_life.csv, clean_mort.csv, clean_gdp.csv  ../data/clean_joined.csv

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
sns.set_theme(style="whitegrid")

In [4]:
# Load the three cleaned files from notebook 
life = pd.read_csv(r"C:\Users\admin\Desktop\Captoneproject-pydata\data\life-expectancy.csv")
mort = pd.read_csv(r"C:\Users\admin\Desktop\Captoneproject-pydata\data\child-mortality.csv")
gdp = pd.read_csv(r"C:\Users\admin\Desktop\Captoneproject-pydata\data\gdp-per-capita-maddison-project-database.csv")


In [5]:
life.head()

,Entity,Code,Year,Life expectancy
0,Afghanistan,AFG,1950,28.1563
1,Afghanistan,AFG,1951,28.5836
2,Afghanistan,AFG,1952,29.0138
3,Afghanistan,AFG,1953,29.4521
4,Afghanistan,AFG,1954,29.6975


In [6]:
mort.head()

,Entity,Code,Year,Under-five mortality rate (selected)
0,Afghanistan,AFG,1957,36.82
1,Afghanistan,AFG,1958,36.21
2,Afghanistan,AFG,1959,35.64
3,Afghanistan,AFG,1960,35.06
4,Afghanistan,AFG,1961,34.52


In [7]:
gdp.head()

,Entity,Code,Year,GDP per capita,GDP per capita (Annotations)
0,Afghanistan,AFG,1950,1156.0,NaN
1,Afghanistan,AFG,1951,1170.0,NaN
2,Afghanistan,AFG,1952,1189.0,NaN
3,Afghanistan,AFG,1953,1240.0,NaN
4,Afghanistan,AFG,1954,1245.0,NaN


In [8]:
keys = ["Entity", "Code", "Year"]
print("Rows before joining:")
print("  life expectancy:", len(life))
print("  child mortality:", len(mort))
print("  GDP per capita: ", len(gdp))

Rows before joining:
  life expectancy: 21565
  child mortality: 17066
  GDP per capita:  21586


In [9]:
   # rename, keep countries, keep needed column
    
rename_map = {
    "Life expectancy": "life_exp",
    "Under-five mortality rate (selected)": "child_mort",
    "GDP per capita": "gdp_pc",
}
life = life.rename(columns=rename_map)
mort = mort.rename(columns=rename_map)
gdp = gdp.rename(columns=rename_map)


# keep real countries only
# Regions and groups (World, Africa, and so on) have no country code, or a code starting with "OWID"
def countries_only(d):
    return d[d["Code"].notna() & ~d["Code"].astype(str).str.startswith("OWID")]

life, mort, gdp = countries_only(life), countries_only(mort), countries_only(gdp)


# keep only the columns needed for the join and the analysis
# (this drops the nearly empty GDP annotations column)
life = life[["Entity", "Code", "Year", "life_exp"]]
mort = mort[["Entity", "Code", "Year", "child_mort"]]
gdp = gdp[["Entity", "Code", "Year", "gdp_pc"]]


# print the row counts after cleaning
print("Rows after keeping countries only:")
print("  life expectancy:", len(life))
print("  child mortality:", len(mort))
print("  GDP per capita: ", len(gdp))

Rows after keeping countries only:
  life expectancy: 19452
  child mortality: 16161
  GDP per capita:  20934


In [10]:
# The three columns to identify one country in one year (the join keys)
keys = ["Entity", "Code", "Year"]

# Join life expectancy with GDP
# how="inner" keeps only the country-years that exist in BOTH tables
step1 = life.merge(gdp, on=keys, how="inner")
print("After life expectancy + GDP:", len(step1), "rows")

# Adding child mortality to the result
joined = step1.merge(mort, on=keys, how="inner")
print("After adding child mortality:", len(joined), "rows")

# Keeping 1950 onward, because coverage is thin and uneven before 1950
before_filter = len(joined)  # remember the count before filtering
joined = joined[joined["Year"] >= 1950].copy()  # .copy() avoids warnings later
print("After the 1950 filter:", len(joined), "rows,", joined["Entity"].nunique(), "countries")

# Checking the brief's requirement of at least 10,000 rows
print("At least 10,000 rows?", len(joined) >= 10000)

# Checking that no region slipped through
print("World rows left (should be 0):", (joined["Entity"] == "World").sum())

# Checking out for missing values and duplicate country-years
print("\nMissing values:")
print(joined[["life_exp", "child_mort", "gdp_pc"]].isna().sum())  # expect 0 in each column
print("Duplicate country-years:", joined.duplicated(["Entity", "Year"]).sum())  # expect 0

After life expectancy + GDP: 13218 rows
After adding child mortality: 11828 rows
After the 1950 filter: 10429 rows, 163 countries
At least 10,000 rows? True
World rows left (should be 0): 0

Missing values:
life_exp      0
child_mort    0
gdp_pc        0
dtype: int64
Duplicate country-years: 0


In [11]:
    # GDP is very right-skewed, so I take the log to make it more even
joined["log_gdp"] = np.log(joined["gdp_pc"])

# child mortality is skewed the same way, so I take its log as well
joined["log_child_mort"] = np.log(joined["child_mort"])

# I group the years into decades so I can compare over time (1987 becomes 1980)
joined["decade"] = (joined["Year"] // 10) * 10

# I split country-years into above and below the median GDP, for my chi-square test
gdp_median = joined["gdp_pc"].median()
joined["gdp_group"] = np.where(joined["gdp_pc"] > gdp_median, "Above median GDP", "Below median GDP")

# I split GDP into four equal groups from poorest to richest, for my ANOVA
joined["gdp_quartile"] = pd.qcut(joined["gdp_pc"], 4, labels=["Q1 poorest", "Q2", "Q3", "Q4 richest"])

# I label each country-year as high or low child mortality compared with the median
mort_median = joined["child_mort"].median()
joined["mort_group"] = np.where(joined["child_mort"] > mort_median, "High child mortality", "Low child mortality")

# I mark each year as before 2000 or 2000 and after, for my A/B-style comparison
joined["period"] = np.where(joined["Year"] < 2000, "Before 2000", "2000 and after")

# I check the first rows of my final table
display(joined.head())
print("Columns:", joined.columns.tolist())

# I save the table so the correlation and test notebooks can use it
joined.to_csv("../data/clean_joined.csv", index=False)
print("Saved data/clean_joined.csv with", len(joined), "rows")

,Entity,Code,Year,life_exp,gdp_pc,child_mort,log_gdp,log_child_mort,decade,gdp_group,gdp_quartile,mort_group,period
0,Afghanistan,AFG,1957,31.3451,1253.0,36.82,7.133296,3.606041,1950,Below median GDP,Q1 poorest,High child mortality,Before 2000
1,Afghanistan,AFG,1958,31.8400,1298.0,36.21,7.168580,3.589335,1950,Below median GDP,Q1 poorest,High child mortality,Before 2000
2,Afghanistan,AFG,1959,32.3365,1307.0,35.64,7.175490,3.573469,1950,Below median GDP,Q1 poorest,High child mortality,Before 2000
3,Afghanistan,AFG,1960,32.7987,1326.0,35.06,7.189922,3.557061,1960,Below median GDP,Q1 poorest,High child mortality,Before 2000
4,Afghanistan,AFG,1961,33.2910,1309.0,34.52,7.177019,3.541539,1960,Below median GDP,Q1 poorest,High child mortality,Before 2000


Columns: ['Entity', 'Code', 'Year', 'life_exp', 'gdp_pc', 'child_mort', 'log_gdp', 'log_child_mort', 'decade', 'gdp_group', 'gdp_quartile', 'mort_group', 'period']


Saved data/clean_joined.csv with 10429 rows


In [12]:
#  reloading the saved file to check it is correct
check = pd.read_csv("../data/clean_joined.csv")
print("Shape:", check.shape)
print("Countries:", check["Entity"].nunique())
print("World rows:", (check["Entity"] == "World").sum())
print("Missing values:", check.isna().sum().sum())
print("Years:", check["Year"].min(), "to", check["Year"].max())

Shape: (10429, 13)
Countries: 163
World rows: 0
Missing values: 0
Years: 1950 to 2022
